In [3]:
import numpy as np
import pandas as pd
from numpy.linalg import norm
from pathlib import Path

def proj_psd(A: np.ndarray) -> np.ndarray:
    """Project a symmetric matrix onto the PSD cone by eigenvalue clipping."""
    w, V = np.linalg.eigh(A)
    A_psd = (V * np.maximum(w, 0.0)) @ V.T
    return 0.5 * (A_psd + A_psd.T)

def higham_nearcorr(A: np.ndarray, tol: float | None = None,
                    max_iterations: int = 100,
                    weights: np.ndarray | None = None) -> np.ndarray:
    """Higham nearest correlation matrix via alternating projections."""
    if not np.allclose(A, A.T, atol=1e-12):
        raise ValueError("Input matrix must be symmetric.")
    n = A.shape[0]
    eps = np.finfo(float).eps
    if tol is None:
        tol = eps * n
    if weights is None:
        weights = np.ones(n)
    W12 = np.sqrt(np.outer(weights, weights))

    X = A.copy()
    Y = A.copy()
    D = np.zeros_like(A)

    rel_diffX = rel_diffY = rel_diffXY = np.inf
    it = 0
    while max(rel_diffX, rel_diffY, rel_diffXY) > tol:
        it += 1
        if it > max_iterations:
            break

        X_old = X.copy()
        R = X - D
        X = proj_psd(W12 * R) / W12
        D = X - R

        Y_old = Y.copy()
        Y = X.copy()
        np.fill_diagonal(Y, 1.0)

        nY = norm(Y, "fro") + eps
        rel_diffX  = norm(X - X_old, "fro") / (norm(X, "fro") + eps)
        rel_diffY  = norm(Y - Y_old, "fro") / nY
        rel_diffXY = norm(Y - X, "fro") / nY

        X = Y.copy()

    return X

def higham_covariance(S_in: pd.DataFrame | np.ndarray,
                      max_iterations: int = 200,
                      tol: float | None = 1e-10) -> pd.DataFrame | np.ndarray:
    """Nearest PSD covariance via: scale to correlation -> Higham -> scale back."""
    is_df = isinstance(S_in, pd.DataFrame)
    A = S_in.to_numpy(float) if is_df else np.array(S_in, float)
    A = 0.5 * (A + A.T)

    var = np.clip(np.diag(A), 0.0, None)
    sd = np.sqrt(np.where(var < 1e-18, 1e-18, var))

    R = (A / sd[:, None]) / sd[None, :]
    R_h = higham_nearcorr(R, tol=tol, max_iterations=max_iterations)

    S_h = (R_h * sd[:, None]) * sd[None, :]
    S_h = 0.5 * (S_h + S_h.T)

    return (pd.DataFrame(S_h, index=S_in.index, columns=S_in.columns)
            if is_df else S_h)

# Read data and give output
DATA_DIR = Path.cwd() / "testfiles_" / "data"
csv_path = DATA_DIR / "testout_1.3.csv"

S = pd.read_csv(csv_path)
for c in S.columns:
    S[c] = pd.to_numeric(S[c], errors="coerce")
if S.shape[0] != S.shape[1]:
    raise ValueError(f"Matrix must be square, got {S.shape}")
S.index = S.columns

Sh = higham_covariance(S, max_iterations=200, tol=1e-10)
print(Sh)

          x1        x2        x3        x4        x5
x1  1.173986 -0.623870 -0.294335 -0.057677 -0.693888
x2 -0.623870  1.318197  0.016449  0.448579  0.143703
x3 -0.294335  0.016449  0.918102  0.354067  0.246866
x4 -0.057677  0.448579  0.354067  0.894764 -0.217062
x5 -0.693888  0.143703  0.246866 -0.217062  0.522607
